In [4]:
pip install sdv sdmetrics

Note: you may need to restart the kernel to use updated packages.


# Libreria SDV
*Autor: Michelle Villalobos Galicia*

*Fecha: 04 de Junio*

In [1]:
#Importamos librerias
import pandas as pd
import numpy as np
import random

In [2]:
import sdv
import sdmetrics

from sdv.metadata import SingleTableMetadata
from sdv .single_table import GaussianCopulaSynthesizer

In [3]:
from sdmetrics.reports.single_table import QualityReport

In [5]:
print("SDV:", sdv.__version__)

SDV: 1.37.0


In [6]:
#Creamos un DataSet (original) que servira como fuente para crear los Datos Sintéticos
dfClientes = pd.DataFrame(
    {
        "cliente_id" : [1,2,3,4,5,6,7,8,9,10],
        "edad" : [23,33,43,28,53,56,43,56,65,40],
        "ingreso_mensual" : [25000,15000,23000,20000,45000,6000,12000,35000,7500,8000],
        "ciudad" : ["Veracruz","Córdoba","Paso del Macho","Amátlan","Fortín","Yanga","Córdoba","Cuitlahuac","Orizaba","Córdoba"]
    }
)

In [7]:
dfClientes.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,1,23,25000,Veracruz
1,2,33,15000,Córdoba
2,3,43,23000,Paso del Macho
3,4,28,20000,Amátlan
4,5,53,45000,Fortín


In [8]:
#Definir los metadatos
metadata = SingleTableMetadata()

In [9]:
metadata.detect_from_dataframe(
    data = dfClientes
)

In [10]:
#Instalar el completo graphviz (URL/winget)
metadata.visualize()

C:\Users\miche\anaconda3\envs\Extraccion9B\Lib\site-packages\sdv\metadata\visualization.py:131: RuntimeWarning: Graphviz does not seem to be installed on this system. For full metadata visualization capabilities, please make sure to have its binaries propertly installed: https://graphviz.gitlab.io/download/
  warnings.warn(warning_message, RuntimeWarning)


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [11]:
metadata.to_dict()

{'primary_key': 'cliente_id',
 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}}}

In [12]:
#Guardamos el metadata en un archivo JSON
metadata.save_to_json(
    "dfClientes_metadata.json"
)

In [13]:
#Entrenamos el modelo para generar los datos sintéticos
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

C:\Users\miche\anaconda3\envs\Extraccion9B\Lib\site-packages\sdv\single_table\base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)


In [14]:
#Entrenamiento
synthesizer.fit(
    dfClientes
)

In [15]:
#Generamos los datos sintéticos
clientes_sinteticos = synthesizer.sample(
    num_rows = 100
)

In [16]:
clientes_sinteticos.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,16169768,55,8473,Yanga
1,4918803,43,30044,Fortín
2,1900081,35,17121,Amátlan
3,531516,30,29035,Yanga
4,10211768,52,14246,Cuitlahuac


In [17]:
clientes_sinteticos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   cliente_id       100 non-null    int64 
 1   edad             100 non-null    int64 
 2   ingreso_mensual  100 non-null    int64 
 3   ciudad           100 non-null    object
dtypes: int64(3), object(1)
memory usage: 3.3+ KB


In [18]:
clientes_sinteticos.describe(include = "all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Córdoba
freq,NaN,NaN,NaN,26
mean,8.532170e+06,43.010000,21841.980000,NaN
std,4.687773e+06,11.788965,11725.486732,NaN
min,4.660500e+04,28.000000,7543.000000,NaN
25%,5.008268e+06,32.000000,10448.500000,NaN
50%,8.656048e+06,41.500000,20935.000000,NaN
75%,1.256688e+07,54.250000,29364.000000,NaN


In [19]:
#DataFrame contaminado
dfClientesGIGO = clientes_sinteticos.copy()

In [20]:
#Colocar edades imposibles
indices = random.sample(list(dfClientesGIGO.index),5)

In [22]:
dfClientesGIGO.loc[indices, "edad"] = -5

In [23]:
#Introducimos valores duplicados
duplicados = dfClientesGIGO.sample(10, random_state = 42)

In [24]:
dfClientesGIGO = pd.concat([dfClientesGIGO, duplicados], ignore_index = True)

In [25]:
#Verificamos valores nulos
dfClientesGIGO.duplicated().sum()

np.int64(10)

In [26]:
dfClientesGIGO.describe(include = "all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Córdoba
freq,NaN,NaN,NaN,28
mean,8.614045e+06,39.463636,22407.818182,NaN
std,4.775996e+06,15.664883,11699.306000,NaN
min,4.660500e+04,-5.000000,7543.000000,NaN
25%,4.948624e+06,29.000000,10736.000000,NaN
50%,8.956428e+06,39.000000,22615.500000,NaN
75%,1.274153e+07,52.750000,29526.000000,NaN


In [27]:
#Generamos el reporte
reporte = QualityReport()

In [28]:
reporte.generate(
    real_data = dfClientes,
    synthetic_data = clientes_sinteticos,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 238.74it/s]|
Column Shapes Score: 80.33%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 63.49it/s]|
Column Pair Trends Score: 23.5%

Overall Score (Average): 51.92%



In [29]:
reporte.generate(
    real_data = dfClientes,
    synthetic_data = dfClientesGIGO,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 353.42it/s]|
Column Shapes Score: 77.88%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 99.60it/s]|
Column Pair Trends Score: 13.64%

Overall Score (Average): 45.76%

